In [2]:
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.abstract_event_listener import AbstractEventListener
from selenium.webdriver.support.events import EventFiringWebDriver, AbstractEventListener
from selenium.webdriver import ActionChains
from selenium.webdriver.common.keys import Keys
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import UnexpectedAlertPresentException
from selenium import webdriver
# from webdriver_auto_update.chrome_app_utils import ChromeAppUtils
# from webdriver_auto_update.webdriver_manager import WebDriverManager
import traceback

#* required
import base64
import re
import os #* ไม่ได้
import win32print
import win32api
import win32gui
import winreg

import time
import subprocess
import threading

import pandas as pd

#* setup
def setup_chrome():
    options = Options()
    options.add_experimental_option("debuggerAddress", "localhost:8989")
    driver = webdriver.Chrome(service=Service(r'C:\bin\chromedriver.exe'), options=options)
    return driver

def get_tabs():
    global merged_dict
    try:
        # if parent.winfo_exists():
        if True:
            print("รายงานจำนวนtabs")

            # * เก็บชื่อ title และ value ของ tab ที่เปิดอยู่
            title_list = []
            # title_list_Idx = [] #!เหมือนจะไม่ได้ใช้
            value_list = []
            # title_dict = {} #!เหมือนจะไม่ได้ใช้
            for idx, handle in enumerate(driver.window_handles):
                driver.switch_to.window(handle)
                # title_list_Idx.append(
                #     driver.title + "["+str(idx)+"]") #!เหมือนจะไม่ได้ใช้
                title_list.append(driver.title)

                value_list.append(driver.current_window_handle)
                # title_dict.update(
                #     {driver.title: driver.current_window_handle}) #!เหมือนจะไม่ได้ใช้

            # * เอาtitle มาทำให้ unique เพราะ title จะสามารถที่จะซ้ำกันได้
            unique_titles = []
            counter = {}
            for item in title_list:
                if item in counter:
                    counter[item] += 1
                    print("counter[item] คือไร: ", counter[item])
                    unique_titles.append(f"{item}{counter[item]-1}")
                else:
                    counter[item] = 1
                    unique_titles.append(item)

            # * เอาList มารวมกัน
            merged_dict = dict(zip(unique_titles, value_list))
            print("มี tabs ไรบ้าง", merged_dict)
            
    except Exception as e:
        traceback_str = traceback.format_exc()
        print(f"An error occirred: {e}")
        print(traceback_str)
        
def read_shop_names_from_excel(file_path):
    # อ่านไฟล์ Excel
    df = pd.read_excel(file_path)
    
    # ดึงค่าจากคอลัมน์ SHOP NAME และแปลงเป็น list
    shop_names = df['SHOP NAME'].dropna().tolist()
    
    return shop_names
    
        
def submit_form(data):
    def wait_element(xpath, text=None):
        element = ""
        while True:
            print("initial")
            try:
                print("try", xpath)
                element = driver.find_element(By.XPATH, xpath)
                print("element: ", element)
                
            except:
                print("continue")
                continue
    
            
            element = driver.find_element(By.XPATH, xpath)
            if element.is_displayed():
                print("element.is_displayed()")
                if not text:
                    print("not text")
                    return element
                    
                elif text in element.text :
                    return element
                else:
                    time.sleep(0.75)
            else:
                print("not element.is_displayed()")
                time.sleep(0.75)
                
    def insert_input(xpath, input):
        target_element = driver.find_element(By.XPATH, xpath)
        target_element.send_keys(Keys.CONTROL, "a")  # เลือกข้อความทั้งหมด
        
        target_element.send_keys(Keys.DELETE)        # ลบข้อความ
        target_element.send_keys(input)
    
    pattern = r'^([A-Z0-9]+)\s(.+)$'
    times = 0 
    
    for text in data:
        try:
            match = re.match(pattern, text)
            if match:
                code, description = match.groups()
                print(f"Code: {code}, Description: {description}")
                if times >= 20:
                    driver.execute_script("window.open('https://crm-plus-backofficesso.buzzebees.com/UserPermission', '_blank');")
                    # สลับไปยังแท็บใหม่
                    driver.switch_to.window(driver.window_handles[-1])
                    
                    # ปิดแท็บเดิม
                    driver.switch_to.window(driver.window_handles[-2])
                    driver.close()

                    # กลับไปที่แท็บใหม่
                    driver.switch_to.window(driver.window_handles[-1])
                    
                    #* reset loop
                    times = 0
                
                while True:
                    try:
                        driver.find_element(By.XPATH, '/html/body/div/div[1]/div[2]/div/div[1]/div[2]/button').click()
                        break
                    except:
                        continue
                
                while True:
                    try:
                        driver.find_element(By.XPATH, '/html/body/div/div[1]/div[3]/div[2]/div[2]/div/label[3]/span[1]/input').click()
                        break
                    except:
                        continue
                
                insert_input('/html/body/div/div[1]/div[3]/div[2]/div[3]/div[1]/input', f"{code}")
                insert_input('/html/body/div/div[1]/div[3]/div[2]/div[3]/div[2]/input', f"{description}")
                
                insert_input('/html/body/div/div[1]/div[3]/div[2]/div[3]/div[5]/input', f"{code.lower()}itcity")
                
                insert_input('/html/body/div/div[1]/div[3]/div[2]/div[3]/div[9]/span/input', "itcity2025")
                time.sleep(0.15)
                insert_input('/html/body/div/div[1]/div[3]/div[2]/div[3]/div[10]/span/input', "itcity2025")
                
                insert_input('/html/body/div/div[1]/div[3]/div[2]/div[3]/div[12]/input', f"{code.lower()}itc.itcity@itcity.co.th")
                insert_input('/html/body/div/div[1]/div[3]/div[2]/div[3]/div[13]/input', "0000000000")
                
                insert_input('/html/body/div/div[1]/div[3]/div[2]/div[4]/div[1]/div[2]/span/input', f"{code} {description}")
                branch_filter = driver.find_element(By.XPATH, '/html/body/div/div[1]/div[3]/div[2]/div[4]/div[1]/div[2]/span/input')
                branch_filter.send_keys(Keys().ENTER)

                while True:
                    try:
                        if not driver.find_element(By.XPATH, "/html/body/div/div[1]/div[3]/div[2]/div[4]/div[2]/div/div/div[5]/label/span[2]").is_displayed():
                            print("display!!")
                            break
                        else:
                            continue
                    except Exception as err:
                        try:
                            print("error display branches",err)
                            print("many branches disappeared")
                            checkbox_label = driver.find_element(By.XPATH, f"//label[span[contains(text(), '{code}')]]")
                            checkbox_label.click()
                            # driver.find_element(By.XPATH, '/html/body/div/div[1]/div[3]/div[2]/div[4]/div[2]/div/div/div/label').click() #*check
                            break
                        except:
                            continue
                
                print("loop done")
                driver.find_element(By.XPATH, '/html/body/div/div[1]/div[3]/div[2]/div[5]/button').click()
                time.sleep(1)
                while True:
                    try:
                        driver.find_element(By.XPATH, '/html/body/div/div[1]/div[5]/div/div[1]/img')
                        print(f"มี error: {code} {description}")
                        driver.get('https://crm-plus-backofficesso.buzzebees.com/UserPermission')
                        break
                    except:
                        print("ไม่ผิดเจอ error")
                        print("มีปุ่มปิดไหม")
                        try:
                            
                            driver.find_element(By.XPATH, '/html/body/div/div[1]/div[4]/div[2]/div[2]/div[2]/button').click()
                            print(f"Complete: {code} {description}")
                            break
                        except:
                            print("ไม่มี")
                            continue
                times += 1
        except Exception as err:
            print(f"Error: {code} {description}")
            
            
    
    
    # print(driver.find_element(By.XPATH, '/html/body/div/div[1]/div[5]/div/div[1]/img').is_displayed())
    
        

#Todo Initialization
driver = setup_chrome()
get_tabs()

#Todo เอา functions ที่ต้องการเทสมาใส่ข้างล่างนี่
driver.switch_to.window(merged_dict['CRM Plus Backoffice'])
current_url = driver.current_url
print("Current URL:", current_url)
file_path = 'CRM_branch.xlsx'
shop_names = read_shop_names_from_excel(file_path)
print("shop_names: ", shop_names)
submit_form(shop_names)

#* เปิดปิด tab เหมือนจะไม่ช่วยลด memory อาจจะต้องลองปิด webdriver 




รายงานจำนวนtabs
มี tabs ไรบ้าง {'CRM Plus Backoffice': '056931DB8DDE58825D783DE5E90E2E12'}
Current URL: https://crm-plus-backofficesso.buzzebees.com/UserPermission
shop_names:  ['S0369 CSC บิ๊กซี ท่าอิฐ ชั้น1', 'S0370 SAMSUNG โคลีเซี่ยม ยะลา ชั้น1', 'S0373 OPPO โรบินสัน บ่อวิน ชั้น2', 'S0374 OPPO โรบินสัน ลพบุรี ชั้น2', 'S0376 CSC ศูนย์การค้าอัศวรรณช้อปปิ้งคอมเพล็กซ์ วัน (หนองคาย)', 'S0377 CSC โลตัส พัทยาเหนือ', 'S0380 OPPO มาร์เก็ตวิลเลจ หัวหิน ชั้น3', 'S0382 OPPO โลตัส หนองบัวลําภู ชั้น1', 'S0386 IT CITY โลตัส ศรีนครินทร์', 'S0390 CSC มาบุญครอง เซ็นเตอร์', 'S0391 CSC สหไทย ทุ่งสง นครศรีธรรมราช', 'S0392 IT CITY บิ๊กซี พระราม 2', 'S0393 it. เสริมไทย คอมเพล็กซ์ มหาสารคาม', 'S0394 CSC โลตัส บึงกาฬ', 'S0395 CSC แฟชั่น ไอส์แลนด์', 'S0396 ACE เซ็นทรัล ขอนแก่น', 'S0397 CSC บิ๊กซี นราธิวาส', 'S0398 IT CITY เซ็นทรัล ศรีราชา', 'S0399 CSC เซ็นทรัล ศรีราชา', 'S0400 ACE เซ็นทรัล ศรีราชา', 'S0402 CSC บิ๊กซี ติวานนท์', 'S0403 CSC บิ๊กซี บ่อวิน ชลบุรี', 'S0404 OPPO บิ๊กซี มหาชัย', 'S0405 IT CITY เซ็น